# Reference discovery & selection — every access pattern

Given Items that carry **multiple** virtual references (kerchunk@CEDA,
icechunk@OSN, icechunk@AWS-S3), demonstrate how a client **discovers**, **filters**,
**selects** (prefer icechunk over kerchunk; prefer S3 over OSN) and **reads** them.

We try **every access pattern** found in the ESGF intel + Slack. Patterns that do
**not** work are written out with the captured error — that is useful feedback, not
a failure of the notebook. A summary matrix is printed at the end.

> Run `build_and_seed_playground.ipynb` first to populate the Playground.

In [ ]:
import traceback

import httpx
import icechunk as ic
import pystac
import xarray as xr

from cmip7_virtualization import select_reference
from cmip7_virtualization.references import is_reference_asset

LOCAL_STAC = "http://localhost:9010"
COLLECTION = "CMIP6"

RESULTS = {}  # pattern -> (status, note)
def record(pattern, status, note=""):
    RESULTS[pattern] = (status, note)
    print(f"{'✅' if status=='WORKS' else '❌' if status=='FAIL' else 'ℹ️ '} {pattern}: {status} {note}")

## Pattern 1 — STAC API search / filter

In [ ]:
items = httpx.get(f"{LOCAL_STAC}/collections/{COLLECTION}/items?limit=20", timeout=30).json()["features"]
print(f"{len(items)} items in the catalog")
item = items[0]
refs = {k: a for k, a in item["assets"].items() if is_reference_asset(a)}
print("reference assets on", item["id"], ":")
for k, a in refs.items():
    print("  ", k, "|", a.get("type"), "| storage=", a.get("cmip7:storage"))
record("1. STAC API search/filter", "WORKS", f"{len(refs)} refs on first item")

## Pattern 2 — pystac `get_assets` + `select_reference` policy

In [ ]:
try:
    pitem = pystac.Item.from_dict(item)
    # native pystac media-type filter
    ic_assets = pitem.get_assets(media_type="application/vnd.zarr+icechunk")
    print("icechunk assets via get_assets:", list(ic_assets))
    # our policy: icechunk over kerchunk, s3 over osn
    key, asset = select_reference({k: a.to_dict() for k, a in pitem.assets.items()})
    print("select_reference picked:", key, "->", asset["href"])
    record("2. pystac get_assets + select_reference", "WORKS", key)
except Exception as e:
    traceback.print_exc(); record("2. pystac get_assets + select_reference", "FAIL", repr(e))

## Pattern 3 — direct Icechunk open (the reliable interim path)

In [ ]:
def source_auth(item):
    """authorize_virtual_chunk_access for the source data hosts of an item."""
    auth = {}
    for a in item["assets"].values():
        if a.get("type") == "application/netcdf":
            pref = "/".join(a["href"].split("/")[:3]) + "/"
            auth[pref] = ic.s3_anonymous_credentials() if pref.startswith("s3://") else None
    return auth


def open_icechunk_asset(item, asset):
    so = asset.get("xarray:storage_options", {})
    bucket = asset["href"].split("/")[2]
    prefix = "/".join(asset["href"].split("/")[3:])
    endpoint = so.get("endpoint_url")
    storage = ic.s3_storage(
        bucket=bucket, prefix=prefix, region=so.get("region", "us-east-1"),
        anonymous=True, endpoint_url=endpoint, force_path_style=bool(endpoint),
    )
    repo = ic.Repository.open(storage=storage, authorize_virtual_chunk_access=source_auth(item))
    return xr.open_zarr(repo.readonly_session("main").store)


for k, a in refs.items():
    if a.get("type") == "application/vnd.zarr+icechunk":
        try:
            ds = open_icechunk_asset(item, a)
            print(f"  {k}: opened {dict(ds.sizes)}")
            record(f"3. direct Icechunk open [{a.get('cmip7:storage')}]", "WORKS", k)
        except Exception as e:
            print(f"  {k}: {type(e).__name__}: {str(e)[:160]}")
            record(f"3. direct Icechunk open [{a.get('cmip7:storage')}]", "FAIL", f"{type(e).__name__}: {str(e)[:100]}")

## Pattern 4 — xpystac `engine="stac"` (the target UX)

Expected (per verified xpystac source, S3-only, no HTTP virtual-chunk path):
- **AWS-S3 store + esgf-world-S3 source** — may **WORK**.
- **OSN store** (custom endpoint) — **FAIL** (xpystac builds `s3_store(region=...)` only).
- **any CEDA-HTTP-sourced store** — **FAIL** (HTTP virtual chunks unsupported).

In [ ]:
import xpystac  # noqa: F401  (registers the 'stac' engine)

for k, a in refs.items():
    if a.get("type") == "application/vnd.zarr+icechunk":
        try:
            asset_obj = pystac.Asset.from_dict(a)
            ds = xr.open_dataset(asset_obj, engine="stac", chunks={})
            print(f"  {k}: opened {dict(ds.sizes)}")
            record(f"4. xpystac engine=stac [{a.get('cmip7:storage')}]", "WORKS", k)
        except Exception as e:
            print(f"  {k}: {type(e).__name__}: {str(e)[:160]}")
            record(f"4. xpystac engine=stac [{a.get('cmip7:storage')}]", "FAIL", f"{type(e).__name__}: {str(e)[:100]}")

## Pattern 5 — kerchunk `reference_file` (CEDA legacy asset)

In [ ]:
ref = item["assets"].get("reference_file")
if not ref:
    record("5. kerchunk reference_file", "N/A", "first item has no kerchunk reference_file")
else:
    try:
        ds = xr.open_dataset(
            "reference://", engine="zarr", chunks={},
            backend_kwargs={"consolidated": False,
                            "storage_options": {"fo": ref["href"], "remote_protocol": "https"}},
        )
        print("opened", dict(ds.sizes))
        record("5. kerchunk reference_file", "WORKS", ref["href"])
    except Exception as e:
        print(f"{type(e).__name__}: {str(e)[:200]}")
        record("5. kerchunk reference_file", "FAIL", f"{type(e).__name__}: {str(e)[:120]}")

## Pattern 6 — intake-ESGF

In [ ]:
try:
    import intake_esgf  # noqa: F401
    record("6. intake-ESGF", "TODO", "installed — STAC/icechunk reference support to be checked")
except ImportError:
    record("6. intake-ESGF", "N/A", "not installed (pip install intake-esgf); icechunk refs unlikely supported")

## Pattern 7 — alternate-assets vs separate-assets (and replica discovery)

We model each virtual store as a **separate top-level asset**, not an `alternate`.
The [alternate-assets extension](https://github.com/stac-extensions/alternate-assets)
requires alternate URLs to point to the **identical file** ("same checksum and file
size"). Two Icechunk stores (OSN vs S3) are *different objects* → not alternates.
This also means esgadd's nesting of icechunk under
`/assets/reference_file/alternate/<site>` is semantically off for virtual stores.

**Where alternate-assets *is* relevant (future):** discovering **replicated source
data**. If the same NetCDF files are mirrored across nodes, those mirrors are true
alternates of each other (identical files), and a virtualization service could read
the `alternate` list to find a replica on node B — then build a *new* virtual store
that points at node B's copy (we would not "mirror" a virtual store pointing at node
A). Worth exploring once replicas are represented in the catalog.

## Summary matrix

In [ ]:
print(f"{'pattern':<48} {'status':<6} note")
print("-" * 90)
for pattern, (status, note) in RESULTS.items():
    print(f"{pattern:<48} {status:<6} {note}")